# Computación científica en ambientes de HPC

En este módulo, estudiaremos como integrar:
* Matemáticas aplicadas,
* programación compilada y
* computación de alto rendimiento

para resolver un problema de interés de las ciencias naturales.
Nuestra plataforma será el ambiente de supercómputo de la UCR: [Cluster Institucional UCR](https://hpc.ucr.ac.cr)

## La ecuación de calor en dos dimensiones

La ecuación de calor describe cómo evoluciona la temperatura en un medio a medida que el calor se difunde espacialmente.

Supongamos que:

* $T(x,y,t)$ representa el campo de temperatura en el espacio-tiempo,
* $x$ y $y$ son coordenadas espaciales,
* $t$ es el tiempo,
* $D$ es el coeficiente de difusión térmica.

La ecuación diferencial parcial que modela este proceso es:
$$
\frac{\partial T}{\partial t} = D \left(\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}\right)
$$

Esta ecuación expresa que la temperatura cambia en el tiempo debido a diferencias locales de temperatura produciendo un flujo de calor desde regiones calientes hacia regiones frías (curiosamente, se puede derivar directamente desde la teoría de la probabilidad sin recurrir a *ningún* argumento físico, más de esto más adelante).

## Discretización espacio-temporal

Para resolver numéricamente la ecuación de calor, podemos discretizamos tanto el espacio como el tiempo.

Definimos una grilla bidimensional:

- $x_i = i h$
- $y_j = j h$

donde:

- $h$ es la separación entre puntos de la grilla,
- $i,j$ son índices enteros.

Además, discretizamos el tiempo:

- $t_n = n \Delta t$

donde:

- $\Delta t$ es el paso temporal.

La temperatura en cada punto de la grilla se representa como:
$$
T_{i,j}^n \equiv T(x_i,y_j,t_n)
$$

Es decir:

- $i,j$ indican la posición espacial,
- $n$ indica el instante temporal.

## Aproximaciones por diferencias finitas

Aproximamos las derivadas espaciales mediante diferencias finitas centrales. La segunda derivada respecto a $x$ se aproxima como:
$$
\frac{\partial^2 T}{\partial x^2} \approx \frac{T_{i+1,j}^n - 2T_{i,j}^n + T_{i-1,j}^n}{h^2}
$$
Análogamente, para la dirección $y$:

$$
\frac{\partial^2 T}{\partial y^2} \approx \frac{T_{i,j+1}^n - 2T_{i,j}^n +  T_{i,j-1}^n}{h^2}
$$

La derivada temporal se aproxima mediante una diferencia hacia adelante:
$$
\frac{\partial T}{\partial t} \approx \frac{T_{i,j}^{n+1} - T_{i,j}^{n}}{\Delta t}
$$

## Regla de actualización explícita

Sustituyendo las aproximaciones discretas en la ecuación de calor obtenemos:
$$
\boxed{T_{i,j}^{n+1} = T_{i,j}^{n} + \alpha \left(T_{i+1,j}^{n} + T_{i-1,j}^{n} + T_{i,j+1}^{n} + T_{i,j-1}^{n} - 4T_{i,j}^{n}\right)}
$$
donde definimos:
$$
\alpha = \frac{D\Delta t}{h^2}
$$

Esta ecuación constituye la regla de actualización utilizada en la simulación numérica. La interpretación física es la siguiente:
- cada punto intercambia energía con sus vecinos,
- la temperatura evoluciona suavizando diferencias locales,
- el calor fluye desde regiones de mayor a menor temperatura.

La regla de actualización genera un *stencil* de cinco puntos:
$$
\begin{matrix}
 & T_{i,j+1} & \\
T_{i-1,j} & T_{i,j} & T_{i+1,j} \\
 & T_{i,j-1} &
\end{matrix}
$$

## Implementación

La sección más costosa computacionalmente corresponde a la rutina que actualiza la grilla al tiempo $t + \Delta t$ con respecto a la grilla espacial al tiempo $t$. Note que este procedimiento se realiza $n = 1, 2, \cdots, N$ veces, por lo cual acelerar cada una de las iteraciones da lugar a un beneficio computacional muy alto.

En `C++`, dicha rutina se vería, con una estructura de datos dos dimensional, similar a:

```c++
for(int i = 1; i < Nx - 1; ++i){
    for(int j = 1; j < Ny - 1; ++j){

        T_new[i][j] = T[i][j] + alpha*(
                                       T[i + 1][j]
                                       + T[i - 1][j]
                                       + T[i][j + 1]
                                       + T[i][j - 1]
                                       - 4.0 * T[i][j]);
    }
}
```

Note que hemos asumido que el número de puntos de la grilla espacial es `Nx` y `Ny` en las direcciones $x$ y $y$, respectivamente. También es importante notar que **no se actualizan los bordes de la grilla**, asumiendo que las condiciones de frontera se encuentran localizadas en los bordes.

### Distribución de memoria

En `C++`, suele ser una mejor opción usar `elementos contigüos en memoria` para representar las estructuras de datos. De esta forma, a nivel de optimización los elementos de la matrix que representa la grilla espacial se encuentran aledaños en memoria física. 

En computación científica, el rendimiento de un programa no depende únicamente del número de operaciones matemáticas realizadas.
* En muchos casos, el verdadero cuello de botella es el acceso a memoria.
* Por esta razón, entender cómo se almacenan los datos en memoria es fundamental para escribir un programa eficiente.

Los CPUs modernos poseen varios niveles de memoria con diferentes velocidades:

```text
Registros
   ↓
Cache L1
   ↓
Cache L2
   ↓
Cache L3
   ↓
  RAM
```

En el diagrama anterior:
* La velocidad de accesos de memoria incrementa de **arriba hacia abajo**.
* La capacidad incrementa de **abajo hacia arriba**.

El uso óptimo de los recursos de memoria afecta el desempeño de una aplicación científica en una medida muy alta.
* La idea corresponde a **accesar espacios de memoria de forma contigüa**, de esta forma, el compilador tiene la opción de guardar información en espacios de cache, para que el siguiente acceso sea mucho más rápido.
* Cuando este acceso se realiza a un espacio que no se encuentra en el cache, el sistema debe seleccionar otra porción de memoria y moverla al cache, borrando el cache anterior. A esto se le conoce como un **cache miss**.
* Cuando la eficiencia de un programa depende de sus accesos de memoria, y no necesarimente de las operaciones a realizar con esos datos, la eficiencia está **acotada por los accesos de memoria**.
* **De ahí la importancia de un buen diseño de arreglos de datos**.
* Algunos ejemplos de aplicaciones científicas cuya eficiencia es *memory-bounded*:
    - simulaciones de PDEs,
    - métodos de stencil,
    - álgebra lineal,
    - procesamiento de imágenes.

Debido a las razones anteriores, suele ser crítico la forma en que manipulamos nuestros arreglos de datos. En general, suele ser mejor usar estructuras de datos cuyos elementos yacen contigüos en memoria. Una forma de hacer esto en nuestro algoritmo es usar matrices que en memoria se encuentran en forma de vectores, cómo hemos estudiado en este módulo. Entonces, una mejor opción es forzar esta estructura:

```c++
for(int i = 1; i < Nx - 1; ++i){
    for(int j = 1; j < Ny - 1; ++j){

        T_new[i * Ny + j] = T[i * Ny + j] + alpha*(
                                       T[(i + 1) * Ny + j]
                                       + T[(i - 1) * Ny + j]
                                       + T[i * Ny + j + 1]
                                       + T[i * Ny + j - 1]
                                       - 4.0 * T[i * Ny + j]);
    }
}
```
asumiendo que `u` y `u_new` tienen tamaño `Nx * Ny` y la matriz se guarda en el formato `row-major`.

## Dinámica

Ver `heat.cpp`. 

En esta implementación, se usa un perfil Gaussiano de temperatura para la condición inicial y se asume que las fronteras de la grilla se encuentran a una temperatura fija.
* El programa escribe los resultados a la terminal: la mejor idea sería redireccionar estos resultados a un archivo
* El programa `animate.plt` es un script de gnuplot que genera una animación de la dinámica.

<center>
<img src="heat.gif" width="600">
</center>

## Aceleración con paralelismo de memoria compartida

Hemos analizado las razones por las cuales en computación científica de alto rendimiento:
- el patrón de acceso a memoria es tan importante como el algoritmo numérico,
- la organización de datos puede afectar dramáticamente el rendimiento,
- comprender caches y localidad es esencial para escribir programas eficientes en arquitecturas modernas.

A luz de este análisis, debemos decidir **cual sección de nuestro programa se puede realizar de forma concurrente**. *Grosso modo*, esto se reduce a evaluar cual procedimiento iterativo puede realizarse **con la menor cantidad de dependencia de datos**. Es común modificar un algoritmo aunque sea menos eficiente, con el propósito de utilizar uno más manejable a la concurrencia.

En nuestro caso, tenemos dos procedimientos iterativos:
1. La evolución temporal: cada paso temporal corresponde a una iteración
2. La regla de actualización: el diferencial espacial se convierte en una regla que actualiza el valor de cierto punto en el espacio con respecto a sus vecinos de la iteración temporal anterior.

> Con respecto al punto 1., no existe una forma directa de dividir el trabajo de forma concurrente. De hecho, para diferencias finitas esto es imposible. Para conocer el estado del sistema a cierto punto del tiempo, se debe conocer el estado al tiempo anterior. Esto se conoce como **data dependency**.

> Con respecto al punto 2., aquí si podemos realizar el trabajo de forma concurrente. Existen distintas estrategias, pero una muy común es subdividir cada una de las filas de la grilla a cada uno de los hilos, de esta forma, se opera de forma concurrente.

Existen consideraciones importantes con respecto a cual es el método más óptimo para subdivir las operaciones concurrentes, tomando en cuenta la contigüidad y los accesos de memoria.

### Práctica guiada

Estudiemos y despleguemos el programa paralelizado `heat_shared.cpp` en el Cluster UCR.